# Retropropagación temporal

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_recurrent-neural-networks/bptt.ipynb` · [Lección original](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# La retropropagación a través del tiempo
<a id="sec_bptt"></a>

Si usted completó los ejercicios en [Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch), habría visto que el recorte de gradiente es vital para evitar que los gradientes masivos ocasionales de entrenamiento desestabilizador. Insinuamos que los gradientes que explotan provienen de la retropropagación a través de secuencias largas. Antes de introducir un montón de arquitecturas RNN modernas, echemos un vistazo más de cerca a cómo *backpropagation* funciona en modelos de secuencia en detalle matemático. Esperemos que esta discusión traiga algo de precisión a la noción de *vanishing* y *exploding* gradientes. Si usted recuerda nuestra discusión de propagación hacia delante y hacia atrás a través de grafos computacionales cuando introdujimos MLPs en [Referencia sec_backprop](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html#sec-backprop), entonces la propagación hacia delante en RNNs debería ser relativamente directa. La aplicación de backpropagation en RNNs se llama *backpropagation a través del tiempo* [Werbos.1990](https://d2l.ai/chapter_references/zreferences.html). Este procedimiento requiere que expandamos (o desenrollemos) el grafo computacional de un RNN un paso en el tiempo a un tiempo. La RNN no recorrida es esencialmente una red de feeded a través de nuestra red de la red.

Las complicaciones surgen porque las secuencias pueden ser bastante largas. No es inusual trabajar con secuencias de texto que consisten en más de mil tokens. Tenga en cuenta que esto plantea problemas tanto desde un punto de vista computacional (demasiada memoria) y optimización (inestabilidad numérica). La entrada desde el primer paso pasa a través de más de 1000 productos de matriz antes de llegar a la salida, y otros 1000 productos de matriz se requieren para calcular el gradiente. Ahora analizamos lo que puede ir mal y cómo abordarlo en la práctica.

## Análisis de los gradientes en las RNN
<a id="subsec_bptt_analysis"></a>

Comenzamos con un modelo simplificado de cómo funciona un RNN. Este modelo ignora detalles sobre los detalles específicos del estado oculto y cómo se actualiza. La notación matemática aquí no distingue explícitamente escalares, vectores y matrices. Sólo estamos tratando de desarrollar alguna intuición. En este modelo simplificado, denotamos $h_t$ como el estado oculto, $x_t$ como entrada, y $o_t$ como salida en el momento paso $t$. Recordemos nuestras discusiones en
[Referencia subsec_rnn_w_hidden_states](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html#subsec-rnn-w-hidden-states)
que la entrada y el estado oculto pueden ser concatenados antes de ser multiplicados por una variable de peso en la capa oculta. Así, utilizamos $w_\textrm{h}$ y $w_\textrm{o}$ para indicar los pesos de la capa oculta y la capa de salida, respectivamente.

$$\begin{aligned}h_t &= f(x_t, h_{t-1}, w_\textrm{h}),\\o_t &= g(h_t, w_\textrm{o}),\end{aligned}$$

:eqlabel:`eq_bptt_ht_ot`

donde $f$ y $g$ son transformaciones de la capa oculta y la capa de salida, respectivamente. Por lo tanto, tenemos una cadena de valores $\{\ldots, (x_{t-1}, h_{t-1}, o_{t-1}), (x_{t}, h_{t}, o_t), \ldots\}$ que dependen unos de otros a través de computación recurrente. La propagación hacia delante es bastante sencilla. Todo lo que necesitamos es enroscar a través de la $(x_t, h_t, o_t)$ triplica un paso de tiempo a la vez. La discrepancia entre la salida $o_t$ y el objetivo deseado $y_t$ se evalúa a continuación por una función objetiva a través de todos los pasos de tiempo $T$ como

$$L(x_1, \ldots, x_T, y_1, \ldots, y_T, w_\textrm{h}, w_\textrm{o}) = \frac{1}{T}\sum_{t=1}^T l(y_t, o_t).$$

Para la retropropagación, las cosas son un poco más difíciles, especialmente cuando calculamos los gradientes con respecto a los parámetros $w_\textrm{h}$ de la función objetivo $L$. Para ser específico, por la regla de la cadena,

$$\begin{aligned}\frac{\partial L}{\partial w_\textrm{h}}  & = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_\textrm{h}}  \\& = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_\textrm{o})}{\partial h_t}  \frac{\partial h_t}{\partial w_\textrm{h}}.\end{aligned}$$

:eqlabel:`eq_bptt_partial_L_wh`

El primero y el segundo factores del producto en [Referencia eq_bptt_partial_L_wh](https://d2l.ai/#eq-bptt-partial-L-wh) son fáciles de calcular. El tercer factor $\partial h_t/\partial w_\textrm{h}$ es donde las cosas se ponen difíciles, ya que necesitamos calcular recurrentemente el efecto del parámetro $w_\textrm{h}$ en $h_t$. De acuerdo con el cálculo recurrente en [Referencia eq_bptt_ht_ot](https://d2l.ai/#eq-bptt-ht-ot), $h_t$ depende de $h_{t-1}$ y $w_\textrm{h}$, donde el cálculo de $h_{t-1}$ también depende de $w_\textrm{h}$. Así, evaluar el derivado total de $h_t$ con respecto a $w_\textrm{h}$ utilizando la regla de cadena rendimientos

$$\frac{\partial h_t}{\partial w_\textrm{h}}= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}} +\frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_\textrm{h}}.$$

:eqlabel:`eq_bptt_partial_ht_wh_recur`

Para obtener el gradiente anterior, supongamos que tenemos tres secuencias $\{a_{t}\},\{b_{t}\},\{c_{t}\}$ satisfaciendo $a_{0}=0$ y $a_{t}=b_{t}+c_{t}a_{t-1}$ para $t=1, 2,\ldots$. Entonces para $t\geq 1$, es fácil mostrar

$$a_{t}=b_{t}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t}c_{j}\right)b_{i}.$$

:eqlabel:`eq_bptt_at`

Sustitución de $a_t$, $b_t$ y $c_t$ según

$$\begin{aligned}a_t &= \frac{\partial h_t}{\partial w_\textrm{h}},\\
b_t &= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}}, \\
c_t &= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}},\end{aligned}$$

el cálculo de gradiente en [Referencia eq_bptt_partial_ht_wh_recur](https://d2l.ai/#eq-bptt-partial-ht-wh-recur) satisface $a_{t}=b_{t}+c_{t}a_{t-1}$. Así, por [Referencia eq_bptt_at](https://d2l.ai/#eq-bptt-at), podemos eliminar el cálculo recurrente en [Referencia eq_bptt_partial_ht_wh_recur](https://d2l.ai/#eq-bptt-partial-ht-wh-recur) con

$$\frac{\partial h_t}{\partial w_\textrm{h}}=\frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t} \frac{\partial f(x_{j},h_{j-1},w_\textrm{h})}{\partial h_{j-1}} \right) \frac{\partial f(x_{i},h_{i-1},w_\textrm{h})}{\partial w_\textrm{h}}.$$

:eqlabel:`eq_bptt_partial_ht_wh_gen`

Mientras que podemos utilizar la regla de la cadena para calcular $\partial h_t/\partial w_\textrm{h}$ recursivamente, esta cadena puede conseguir muy largo cuando $t$ es grande. Vamos a discutir una serie de estrategias para tratar con este problema.

### Computación completa

Una idea podría ser calcular la suma completa en [Referencia eq_bptt_partial_ht_wh_gen](https://d2l.ai/#eq-bptt-partial-ht-wh-gen). Sin embargo, esto es muy lento y los gradientes pueden estallar, ya que los cambios sutiles en las condiciones iniciales pueden potencialmente afectar el resultado mucho. Es decir, podríamos ver cosas similares al efecto mariposa, donde los cambios mínimos en las condiciones iniciales conducen a cambios desproporcionados en el resultado. Esto es generalmente indeseable. Después de todo, estamos buscando estimadores robustos que generalicen bien. Por lo tanto, esta estrategia casi nunca se utiliza en la práctica.

### Truncamiento temporal

Alternativamente, podemos truncar la suma en
[Referencia eq_bptt_partial_ht_wh_gen](https://d2l.ai/#eq-bptt-partial-ht-wh-gen)
Después de los pasos $\tau$. Esto es lo que hemos estado discutiendo hasta ahora. Esto conduce a una *aproximación* del verdadero gradiente, simplemente terminando la suma en $\partial h_{t-\tau}/\partial w_\textrm{h}$. En la práctica esto funciona bastante bien. Es lo que comúnmente se conoce como retropropagación truncada a través del tiempo [Jaeger.2002](https://d2l.ai/chapter_references/zreferences.html). Una de las consecuencias de esto es que el modelo se centra principalmente en la influencia a corto plazo en lugar de consecuencias a largo plazo. Esto es realmente *deseable*, ya que sesgina la estimación hacia modelos más simples y más estables.

### Truncamiento aleatorio

Por último, podemos reemplazar $\partial h_t/\partial w_\textrm{h}$ por una variable aleatoria que es correcta en la expectativa, pero trunca la secuencia. Esto se logra utilizando una secuencia de $\xi_t$ con $0 \leq \pi_t \leq 1$ predefinido, donde $P(\xi_t = 0) = 1-\pi_t$ y $P(\xi_t = \pi_t^{-1}) = \pi_t$, por lo tanto $E[\xi_t] = 1$. Utilizamos esto para reemplazar el gradiente $\partial h_t/\partial w_\textrm{h}$ en [Referencia eq_bptt_partial_ht_wh_recur](https://d2l.ai/#eq-bptt-partial-ht-wh-recur) con

$$z_t= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}} +\xi_t \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_\textrm{h}}.$$

De la definición de $\xi_t$ se deduce que $E[z_t] = \partial h_t/\partial w_\textrm{h}$. Siempre que $\xi_t = 0$ el cálculo recurrente termina en ese momento paso $t$. Esto conduce a una suma ponderada de secuencias de longitudes variables, donde las secuencias largas son raras pero con sobrepeso adecuado.
[Tallec.Ollivier.2017](https://d2l.ai/chapter_references/zreferences.html).

### Comparando estrategias
![Estrategias de gradiente en RNN. De arriba abajo: truncamiento aleatorio, truncamiento regular y cálculo completo.](../recursos/originales/truncated-bptt.svg)
<a id="fig_truncated_bptt"></a>

[Referencia fig_truncated_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-truncated-bptt) ilustra las tres estrategias 
al analizar los primeros caracteres de *The Time Machine* usando backpropagation a través del tiempo para RNNs:

* La primera fila es la truncación aleatoria que divide el texto en segmentos de longitudes variables.
* La segunda fila es la truncación regular que rompe el texto en secuencias de la misma longitud. Esto es lo que hemos estado haciendo en experimentos RNN.
* La tercera fila es la retropropagación completa a través del tiempo que conduce a una expresión computacionalmente inviable.

Desafortunadamente, aunque atractivo en teoría, la truncación aleatoria no funciona mucho mejor que la truncación regular, lo más probable debido a una serie de factores. Primero, el efecto de una observación después de una serie de pasos de retropropagación en el pasado es bastante suficiente para capturar dependencias en la práctica. Segundo, la varianza aumentada contrarresta el hecho de que el gradiente es más preciso con más pasos. Tercero, en realidad * queremos * modelos que tienen sólo un corto rango de interacciones. Por lo tanto, regularmente truncada retropropagación a través del tiempo tiene un ligero efecto regularizador que puede ser deseable.

## Retropropagación a través del tiempo en detalle
Después de discutir el principio general, vamos a discutir la retropropagación a través del tiempo en detalle. A diferencia del análisis en [Referencia subsec_bptt_analysis](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#subsec-bptt-analysis), en el siguiente vamos a mostrar cómo calcular los gradientes de la función objetivo con respecto a todos los parámetros del modelo descompuesto. Para mantener las cosas simples, consideramos un RNN sin parámetros de sesgo, cuya función de activación en la capa oculta utiliza la asignación de identidad ($\phi(x)=x$). Para el paso de tiempo $t$, dejar que el único ejemplo de entrada y el objetivo sea $\mathbf{x}_t \in \mathbb{R}^d$ y $y_t$, respectivamente. El estado oculto $\mathbf{h}_t \in \mathbb{R}^h$ y la salida $\mathbf{o}_t \in \mathbb{R}^q$ se computan como

$$\begin{aligned}\mathbf{h}_t &= \mathbf{W}_\textrm{hx} \mathbf{x}_t + \mathbf{W}_\textrm{hh} \mathbf{h}_{t-1},\\
\mathbf{o}_t &= \mathbf{W}_\textrm{qh} \mathbf{h}_{t},\end{aligned}$$

donde $\mathbf{W}_\textrm{hx} \in \mathbb{R}^{h \times d}$, $\mathbf{W}_\textrm{hh} \in \mathbb{R}^{h \times h}$ y $\mathbf{W}_\textrm{qh} \in \mathbb{R}^{q \times h}$ son los parámetros de peso. Denotar por $l(\mathbf{o}_t, y_t)$ la pérdida en el paso de tiempo $t$. Nuestra función objetiva, la pérdida sobre $T$ pasos de tiempo desde el comienzo de la secuencia es así

$$L = \frac{1}{T} \sum_{t=1}^T l(\mathbf{o}_t, y_t).$$

Con el fin de visualizar las dependencias entre las variables y parámetros del modelo durante el cálculo de la RNN, podemos dibujar un grafo computacional para el modelo, como se muestra en [Referencia fig_rnn_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-rnn-bptt). Por ejemplo, el cálculo de los estados ocultos del paso de tiempo 3, $\mathbf{h}_3$, depende de los parámetros del modelo $\mathbf{W}_\textrm{hx}$ y $\mathbf{W}_\textrm{hh}$, el estado oculto del paso de tiempo anterior $\mathbf{h}_2$, y la entrada del paso de tiempo actual $\mathbf{x}_3$.

![Grafo de una RNN con tres pasos temporales. Las cajas representan variables o parámetros sombreados; los círculos representan operadores.](../recursos/originales/rnn-bptt.svg)
<a id="fig_rnn_bptt"></a>

Como se acaba de mencionar, los parámetros del modelo en [Referencia fig_rnn_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-rnn-bptt) son $\mathbf{W}_\textrm{hx}$, $\mathbf{W}_\textrm{hh}$ y $\mathbf{W}_\textrm{qh}$. Generalmente, el entrenamiento de este modelo requiere cálculo de gradiente con respecto a estos parámetros $\partial L/\partial \mathbf{W}_\textrm{hx}$, $\partial L/\partial \mathbf{W}_\textrm{hh}$ y $\partial L/\partial \mathbf{W}_\textrm{qh}$. De acuerdo con las dependencias en [Referencia fig_rnn_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-rnn-bptt), podemos atravesar en la dirección opuesta de las flechas para calcular y almacenar los gradientes a su vez. Para expresar de manera flexible la multiplicación de matrices, vectores y escalares de diferentes formas en la regla de cadena, seguimos utilizando el operador $\textrm{prod}$ como se describe en [Referencia sec_backprop](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html#sec-backprop).

En primer lugar, diferenciar la función objetiva con respecto a la salida del modelo en cualquier momento paso $t$ es bastante sencillo:

$$\frac{\partial L}{\partial \mathbf{o}_t} =  \frac{\partial l (\mathbf{o}_t, y_t)}{T \cdot \partial \mathbf{o}_t} \in \mathbb{R}^q.$$

:eqlabel:`eq_bptt_partial_L_ot`

Ahora podemos calcular el gradiente del objetivo con respecto al parámetro $\mathbf{W}_\textrm{qh}$ en la capa de salida: $\partial L/\partial \mathbf{W}_\textrm{qh} \in \mathbb{R}^{q \times h}$. Basado en [Referencia fig_rnn_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-rnn-bptt), el objetivo $L$ depende de $\mathbf{W}_\textrm{qh}$ a través de $\mathbf{o}_1, \ldots, \mathbf{o}_T$. Usando la regla de cadena produce

$$
\frac{\partial L}{\partial \mathbf{W}_\textrm{qh}}
= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{W}_\textrm{qh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{o}_t} \mathbf{h}_t^\top,
$$

donde $\partial L/\partial \mathbf{o}_t$ es dado por [Referencia eq_bptt_partial_L_ot](https://d2l.ai/#eq-bptt-partial-L-ot).

A continuación, como se muestra en [Referencia fig_rnn_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-rnn-bptt), en el último paso $T$, la función objetivo $L$ depende del estado oculto $\mathbf{h}_T$ sólo a través de $\mathbf{o}_T$. Por lo tanto, podemos encontrar fácilmente el gradiente $\partial L/\partial \mathbf{h}_T \in \mathbb{R}^h$ utilizando la regla de cadena:

$$\frac{\partial L}{\partial \mathbf{h}_T} = \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_T}, \frac{\partial \mathbf{o}_T}{\partial \mathbf{h}_T} \right) = \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_T}.$$

:eqlabel:`eq_bptt_partial_L_hT_final_step`

Se vuelve más difícil para cualquier paso $t < T$, donde la función objetivo $L$ depende de $\mathbf{h}_t$ a través de $\mathbf{h}_{t+1}$ y $\mathbf{o}_t$. De acuerdo con la regla de cadena, el gradiente del estado oculto $\partial L/\partial \mathbf{h}_t \in \mathbb{R}^h$ en cualquier momento paso $t < T$ se puede calcular de forma recurrente como:

$$\frac{\partial L}{\partial \mathbf{h}_t} = \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_{t+1}}, \frac{\partial \mathbf{h}_{t+1}}{\partial \mathbf{h}_t} \right) + \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{h}_t} \right) = \mathbf{W}_\textrm{hh}^\top \frac{\partial L}{\partial \mathbf{h}_{t+1}} + \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_t}.$$

:eqlabel:`eq_bptt_partial_L_ht_recur`

Para el análisis, ampliar el cálculo recurrente para cualquier paso de tiempo $1 \leq t \leq T$ da

$$\frac{\partial L}{\partial \mathbf{h}_t}= \sum_{i=t}^T {\left(\mathbf{W}_\textrm{hh}^\top\right)}^{T-i} \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_{T+t-i}}.$$

:eqlabel:`eq_bptt_partial_L_ht`

Podemos ver en [Referencia eq_bptt_partial_L_ht](https://d2l.ai/#eq-bptt-partial-L-ht) que este simple ejemplo lineal ya exhibe algunos problemas clave de los modelos de secuencias largas: involucra potencialmente poderes muy grandes de $\mathbf{W}_\textrm{hh}^\top$. En él, los valores propios menores de 1 desaparecen y los valores propios mayores de 1 divergen. Esto es numéricamente inestable, que se manifiesta en forma de desapareciendo y explotando gradientes. Una manera de abordar esto es truncar los pasos de tiempo en un tamaño computacionalmente conveniente como se discute en [Referencia subsec_bptt_analysis](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#subsec-bptt-analysis). En la práctica, esta truncación también se puede realizar separando el gradiente después de un número dado de pasos de tiempo. Más adelante, veremos cómo modelos de secuencia más sofisticados como la memoria a corto plazo puede aliviar esto aún más.

Por último, [Referencia fig_rnn_bptt](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html#fig-rnn-bptt) muestra que la función objetivo $L$ depende de los parámetros modelo $\mathbf{W}_\textrm{hx}$ y $\mathbf{W}_\textrm{hh}$ en la capa oculta a través de estados ocultos $\mathbf{h}_1, \ldots, \mathbf{h}_T$. Para calcular gradientes con respecto a tales parámetros $\partial L / \partial \mathbf{W}_\textrm{hx} \in \mathbb{R}^{h \times d}$ y $\partial L / \partial \mathbf{W}_\textrm{hh} \in \mathbb{R}^{h \times h}$, aplicamos la regla de cadena que da

$$
\begin{aligned}
\frac{\partial L}{\partial \mathbf{W}_\textrm{hx}}
&= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_\textrm{hx}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{x}_t^\top,\\
\frac{\partial L}{\partial \mathbf{W}_\textrm{hh}}
&= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_\textrm{hh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{h}_{t-1}^\top,
\end{aligned}
$$

donde $\partial L/\partial \mathbf{h}_t$ que se calcula de forma recurrente por
[Referencia eq_bptt_partial_L_hT_final_step](https://d2l.ai/#eq-bptt-partial-L-hT-final-step)
y [Referencia eq_bptt_partial_L_ht_recur](https://d2l.ai/#eq-bptt-partial-L-ht-recur) es la cantidad clave que afecta la estabilidad numérica.

Puesto que la retropropagación a través del tiempo es la aplicación de la retropropagación en RNNs, como hemos explicado en [Referencia sec_backprop](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html#sec-backprop), el entrenamiento de RNNs alterna la propagación hacia delante con la retropropagación a través del tiempo. Además, la retropropagación a través del tiempo calcula y almacena los gradientes anteriores a su vez. Específicamente, los valores intermedios almacenados se reutilizan para evitar cálculos duplicados, como almacenar $\partial L/\partial \mathbf{h}_t$ para ser utilizado en el cálculo de $\partial L / \partial \mathbf{W}_\textrm{hx}$ y $\partial L / \partial \mathbf{W}_\textrm{hh}$.

## Resumen
La retropropagación a través del tiempo es simplemente una aplicación de la retropropagación a modelos secuenciales con un estado oculto. La truncación, como regular o aleatoria, es necesaria para la comodidad computacional y la estabilidad numérica. Altas potencias de matrices pueden conducir a valores propios divergentes o desvanecientes. Esto se manifiesta en forma de gradientes que explotan o desaparecen. Para la computación eficiente, los valores intermedios se encajonan durante la retropropagación a través del tiempo.



### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «Retropropagación temporal».


## Ejercicios
1. Supongamos que tenemos una matriz simétrica $\mathbf{M} \in \mathbb{R}^{n \times n}$ con valores propios $\lambda_i$ cuyos correspondientes vectores son $\mathbf{v}_i$ ($i = 1, \ldots, n$). Sin pérdida de generalidad, supongamos que se ordenan en el orden $|\lambda_i| \geq |\lambda_{i+1}|$.
   1. Mostrar que $\mathbf{M}^k$ tiene valores propios $\lambda_i^k$.
   1. Demostrar que para un vector al azar $\mathbf{x} \in \mathbb{R}^n$, con alta probabilidad $\mathbf{M}^k \mathbf{x}$ será muy alineado con el vector $\mathbf{v}_1$
de $\mathbf{M}$. Formalizar esta declaración.
   1. ¿Qué significa el resultado anterior para los gradientes en RNNs?
1. Además del recorte de gradiente, ¿puede pensar en otros métodos para hacer frente a la explosión de gradiente en redes neuronales recurrentes?

[Debate del original](https://discuss.d2l.ai/t/334)
